<a href="https://colab.research.google.com/github/GabrielPereira-03/Computabilidade-e-Complexidade-de-Algoritmos/blob/main/Atividade_Teoria_Filas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Atividade de lista de hospital
Aluno: Gabriel Tavares Pereira Silva

In [ ]:
import heapq
from datetime import datetime, timedelta

# --- Configurações Globais ---
TEMPO_CONSULTA_MINUTOS = 20
MAPA_PRIORIDADE = {
    'vermelho': 4,
    'laranja': 3,
    'amarelo': 2,
    'azul': 1,
}
ESPECIALIDADES = [
    'Clínica Médica', 'Pediatria', 'Ortopedia', 'Cardiologia',
    'Ginecologia/Obstetrícia', 'Otorrinolaringologia'
]

# --- Estruturas de Dados ---
filas_especialidades = {esp: [] for esp in ESPECIALIDADES}
proximo_paciente_id = 0  # Contador de ordem de chegada
ultima_consulta_fim = {esp: datetime(1, 1, 1, 8, 0) for esp in ESPECIALIDADES} # Horário fictício inicial (ex: 08:00)

def converter_hora_string(hora_str):
    """Converte 'HH:MM' para objeto datetime. Assume a data de hoje para simplificar."""
    hoje = datetime.now().date()
    h, m = map(int, hora_str.split(':'))
    return datetime(hoje.year, hoje.month, hoje.day, h, m)

def timedelta_to_minutes(td):
    """Converte um timedelta para minutos totais."""
    return td.total_seconds() / 60

# --- Classe/Objeto Paciente ---
class Paciente:
    def __init__(self, nome, idade, cor_urgencia, especialidade, hora_chegada_str, id_chegada):
        self.nome = nome
        self.idade = idade
        self.cor_urgencia = cor_urgencia
        self.especialidade = especialidade
        self.hora_chegada = converter_hora_string(hora_chegada_str)
        self.prioridade = MAPA_PRIORIDADE.get(cor_urgencia.lower(), 1)
        self.id_chegada = id_chegada
        self.hora_inicio_prevista = None
        self.hora_fim_prevista = None

    def __repr__(self):
        return f"({self.nome}, {self.cor_urgencia[:3]}, Chegada: {self.hora_chegada.strftime('%H:%M')})"

# --- Funções de Operação ---

def adicionar_paciente(nome, idade, cor_urgencia, especialidade, hora_chegada_str):
    """
    Insere o paciente na Fila de Prioridade da especialidade. O(log n_s).
    """
    global proximo_paciente_id
    if especialidade not in filas_especialidades:
        print(f"Erro: Especialidade '{especialidade}' não suportada.")
        return

    paciente = Paciente(nome, idade, cor_urgencia, especialidade, hora_chegada_str, proximo_paciente_id)
    proximo_paciente_id += 1

    # Chave para o Min-Heap: (-Prioridade, OrdemChegada, ObjetoPaciente)
    # Negativo na prioridade garante que maior prioridade (Vermelho=4) saia primeiro.
    # OrdemChegada garante FIFO no desempate de prioridade.
    heap_item = (-paciente.prioridade, paciente.id_chegada, paciente)
    heapq.heappush(filas_especialidades[especialidade], heap_item)
    print(f"[{hora_chegada_str}] Paciente {nome} adicionado a {especialidade} com prioridade {cor_urgencia}.")

def proximo(especialidade):
    """
    Remove e retorna o próximo paciente a ser atendido por especialidade. O(log n_s).
    """
    if especialidade not in filas_especialidades or not filas_especialidades[especialidade]:
        return f"Não há pacientes na fila de {especialidade}."

    # Pop do topo do Heap (o de maior prioridade)
    _, _, paciente = heapq.heappop(filas_especialidades[especialidade])

    # Simulação de agendamento no momento da chamada:
    global ultima_consulta_fim
    hora_chegada = paciente.hora_chegada

    # Previsão de Início = max(hora_chegada, fim_da_última_consulta)
    hora_inicio = max(hora_chegada, ultima_consulta_fim[especialidade])
    hora_fim = hora_inicio + timedelta(minutes=TEMPO_CONSULTA_MINUTOS)

    # Atualiza o fim da agenda do médico
    ultima_consulta_fim[especialidade] = hora_fim

    # O paciente que está sendo chamado pode ter sido calculado em uma previsão anterior,
    # mas o cálculo final de tempo de espera/início é feito ao ser chamado.
    tempo_espera = timedelta_to_minutes(hora_inicio - hora_chegada)

    return (f"**CHAMANDO:** {paciente.nome} ({paciente.cor_urgencia}) de {especialidade}. "
            f"Chegada: {hora_chegada.strftime('%H:%M')}, Início: {hora_inicio.strftime('%H:%M')}, "
            f"Fim: {hora_fim.strftime('%H:%M')}. Espera: {max(0, tempo_espera):.0f} min.")


def previsao_atendimento(especialidade):
    """
    Simula o agendamento sequencial para a fila atual. O(n_s).
    """
    if especialidade not in filas_especialidades or not filas_especialidades[especialidade]:
        return f"Fila de {especialidade} vazia."

    # Cria uma cópia da fila (shallow copy) para não modificar a original
    fila_copia = filas_especialidades[especialidade][:]

    # O heap é copiado, mas o algoritmo percorre em ordem (Heap Sort)
    agenda_simulada = []

    # Início da simulação: considera o último horário de fim de consulta real
    prox_inicio_disponivel = ultima_consulta_fim[especialidade]

    # Percorre o heap em ordem de prioridade (O(n_s log n_s) se fosse heap pop, mas
    # podemos percorrê-lo de forma otimizada O(n_s) se ordenarmos a cópia, ou
    # O(n_s log n_s) se fizermos pop). Usaremos o pop em uma cópia para garantir a ordem exata.

    temp_heap = fila_copia[:] # Faz uma cópia da lista de pacientes na fila

    resultados = []
    while temp_heap:
        # Pop da cópia para obter a ordem correta
        _, _, paciente = heapq.heappop(temp_heap)

        # Previsão de Início = max(hora_chegada, fim_da_última_consulta_simulada)
        hora_chegada = paciente.hora_chegada
        hora_inicio = max(hora_chegada, prox_inicio_disponivel)
        hora_fim = hora_inicio + timedelta(minutes=TEMPO_CONSULTA_MINUTOS)

        # Atualiza o próximo início para o simulador
        prox_inicio_disponivel = hora_fim

        resultados.append(
            f"-> {paciente.nome} ({paciente.cor_urgencia}): Chegada {hora_chegada.strftime('%H:%M')}, "
            f"Previsão Início: {hora_inicio.strftime('%H:%M')}, Fim: {hora_fim.strftime('%H:%M')}"
        )

    return "\n".join(resultados)

def tempo_medio_espera(especialidade, hora_atual_str):
    """
    Calcula o tempo médio de espera dos pacientes na fila. O(n_s).
    """
    if especialidade not in filas_especialidades or not filas_especialidades[especialidade]:
        return f"Não há pacientes na fila de {especialidade}."

    hora_atual = converter_hora_string(hora_atual_str)

    # Reutiliza a lógica de previsão para obter o tempo de início previsto
    fila_copia = filas_especialidades[especialidade][:]

    prox_inicio_disponivel = ultima_consulta_fim[especialidade]

    tempos_espera = []
    temp_heap = fila_copia[:]

    while temp_heap:
        _, _, paciente = heapq.heappop(temp_heap)

        hora_chegada = paciente.hora_chegada
        hora_inicio = max(hora_chegada, prox_inicio_disponivel)
        hora_fim = hora_inicio + timedelta(minutes=TEMPO_CONSULTA_MINUTOS)

        prox_inicio_disponivel = hora_fim

        # Tempo de espera = max(0, hora_prevista_início - hora_chegada)
        # O cálculo deve usar o tempo de início previsto
        tempo_espera_minutos = timedelta_to_minutes(hora_inicio - hora_chegada)
        tempos_espera.append(max(0, tempo_espera_minutos))

    if not tempos_espera:
        return f"Não há pacientes a calcular em {especialidade}."

    media_espera = sum(tempos_espera) / len(tempos_espera)
    return (f"Tempo médio de espera previsto em {especialidade}: "
            f"{media_espera:.1f} minutos para a fila atual.")

def listar_fila(especialidade):
    """
    Mostra a fila em ordem de atendimento (ordem do Heap). O(n_s log n_s).
    """
    if especialidade not in filas_especialidades or not filas_especialidades[especialidade]:
        return f"Fila de {especialidade} vazia."

    # Para listar em ordem, precisamos usar o algoritmo de heap sort na cópia
    fila_ordenada = sorted(filas_especialidades[especialidade], key=lambda x: x[0:2])

    lista = [f"--- Fila: {especialidade} ({len(fila_ordenada)} pacientes) ---"]

    for i, (_, _, paciente) in enumerate(fila_ordenada):
        lista.append(
            f"{i+1}. {paciente.nome}, {paciente.idade} anos, {paciente.cor_urgencia} "
            f"(Chegada: {paciente.hora_chegada.strftime('%H:%M')})"
        )
    return "\n".join(lista)

# --- Exemplo de Teste ---

print("--- 1. Cadastro de Pacientes ---")
adicionar_paciente('Maria Silva', 34, 'laranja', 'Ortopedia', '08:05')
adicionar_paciente('João Souza', 62, 'amarelo', 'Clínica Médica', '08:07')
adicionar_paciente('Ana Lima', 5, 'vermelho', 'Pediatria', '08:09')
adicionar_paciente('Carla Dias', 41, 'azul', 'Ortopedia', '08:10')
adicionar_paciente('Pedro Neri', 58, 'laranja', 'Cardiologia', '08:12')
adicionar_paciente('Rui Campos', 29, 'amarelo', 'Ortopedia', '08:14')
adicionar_paciente('Eva Mello', 70, 'laranja', 'Cardiologia', '08:15')
adicionar_paciente('Lucas Gomes', 22, 'azul', 'Clínica Médica', '08:18')

print("\n--- 2. Ordem de Atendimento (Listar Fila) ---")
print(listar_fila('Ortopedia'))
print(listar_fila('Cardiologia'))
print(listar_fila('Pediatria'))

print("\n--- 3. Previsão de Atendimento (Agenda) ---")
print(f"\n--- Agenda Ortopedia ---")
print(previsao_atendimento('Ortopedia'))
print(f"\n--- Agenda Clínica Médica ---")
print(previsao_atendimento('Clínica Médica'))
print(f"\n--- Agenda Cardiologia ---")
print(previsao_atendimento('Cardiologia'))

print("\n--- 4. Tempo Médio de Espera ---")
print(tempo_medio_espera('Ortopedia', '08:30'))
print(tempo_medio_espera('Cardiologia', '08:30'))

print("\n--- 5. Chamar Próximo Paciente ---")
print(proximo('Ortopedia'))  # Maria (laranja)
print(proximo('Ortopedia'))  # Rui (amarelo) - Início: 08:25 (fim da Maria)

print("\n--- 6. Nova Previsão de Atendimento após 2 chamadas ---")
# O último fim real de Ortopedia agora é 08:45
print(f"\n--- Nova Agenda Ortopedia (Fim real: {ultima_consulta_fim['Ortopedia'].strftime('%H:%M')}) ---")
print(previsao_atendimento('Ortopedia')) # Carla (azul) inicia 08:45

print(proximo('Pediatria')) # Ana (vermelho)

--- 1. Cadastro de Pacientes ---
[08:05] Paciente Maria Silva adicionado a Ortopedia com prioridade laranja.
[08:07] Paciente João Souza adicionado a Clínica Médica com prioridade amarelo.
[08:09] Paciente Ana Lima adicionado a Pediatria com prioridade vermelho.
[08:10] Paciente Carla Dias adicionado a Ortopedia com prioridade azul.
[08:12] Paciente Pedro Neri adicionado a Cardiologia com prioridade laranja.
[08:14] Paciente Rui Campos adicionado a Ortopedia com prioridade amarelo.
[08:15] Paciente Eva Mello adicionado a Cardiologia com prioridade laranja.
[08:18] Paciente Lucas Gomes adicionado a Clínica Médica com prioridade azul.

--- 2. Ordem de Atendimento (Listar Fila) ---
--- Fila: Ortopedia (3 pacientes) ---
1. Maria Silva, 34 anos, laranja (Chegada: 08:05)
2. Rui Campos, 29 anos, amarelo (Chegada: 08:14)
3. Carla Dias, 41 anos, azul (Chegada: 08:10)
--- Fila: Cardiologia (2 pacientes) ---
1. Pedro Neri, 58 anos, laranja (Chegada: 08:12)
2. Eva Mello, 70 anos, laranja (Chegada: